# Xem box YOLO và nhãn DenseNet121 trên ảnh tự tải lên

Notebook Colab này dùng đúng file `EXP-DENSENET121-YOLO-COMBINED-TRAINING-034_FULL.zip` trong [thư mục Drive đã cung cấp](https://drive.google.com/drive/folders/1wvWIlG1jsaecMFPiI18-DKk3hyndR92x). Chạy các ô từ trên xuống; dùng thanh **Files** bên trái Colab để tải JPG/PNG/WebP/BMP từ máy tính vào `/content`. Kết quả hiện ngay dưới ô chạy và được lưu trong `/content/exp034_demo/outputs`.

**Ba loại nhãn/độ tin cậy được tách rõ:** YOLO tìm vị trí và đưa ra lớp/điểm detector; DenseNet121 chạy trên **toàn ảnh** để cho nhãn và softmax của bộ 5 seed; mô hình kết hợp F3 cho nhãn/độ tin cậy chính thức của EXP-034. Notebook còn chạy DenseNet trên **crop của từng box** để bạn quan sát nhãn ngay trên box. Crop này là phép thử bổ sung: EXP-034 huấn luyện DenseNet bằng toàn ảnh, nên điểm crop chưa được kiểm định hay hiệu chuẩn như điểm F3. Ảnh bạn tải lên không có nhãn thật, vì vậy notebook không tính accuracy.

?? ph?c v? quan s?t, notebook v? m?i box v??t ng??ng YOLO, k? c? khi F3 d? ?o?n `healthy` ho?c t? ch?i d? ?o?n. Pipeline tri?n khai g?c c? th? ?n box trong c?c tr??ng h?p ??.


In [1]:
# 1. Chuẩn bị Colab và tham số hiển thị
import importlib.util, os, subprocess, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

def ensure_package(module, package):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for module, package in [
    ("googleapiclient", "google-api-python-client"),
    ("ultralytics", "ultralytics"),
    ("tensorflow", "tensorflow"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
]:
    ensure_package(module, package)

import csv, hashlib, io, json, shutil, zipfile
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import display
import tensorflow as tf
from google.colab import auth, files
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

EXPERIMENT_ID = "EXP-DENSENET121-YOLO-COMBINED-TRAINING-034"
DRIVE_FILE_ID = "1XhMt73LKJOzlmxd-di17f6DmdYGh0iF3"
DRIVE_FOLDER_ID = "1wvWIlG1jsaecMFPiI18-DKk3hyndR92x"
ZIP_NAME = EXPERIMENT_ID + "_FULL.zip"
EXPECTED_ZIP_BYTES = 779388067
WORK_DIR = Path("/content/exp034_demo")
ZIP_PATH = WORK_DIR / ZIP_NAME
BUNDLE_DIR = WORK_DIR / "pipeline_bundle"
UPLOAD_DIR = WORK_DIR / "uploads"
OUTPUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, BUNDLE_DIR, UPLOAD_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Ngưỡng box mặc định lấy từ frozen_config.json, được chọn trên YOLO Validation.
BOX_CONF_OVERRIDE = None       # Ví dụ 0.15 để quan sát nhiều box hơn; F3 vẫn dùng box gốc.
CROP_PAD_RATIO = 0.10         # Thêm 10% ngữ cảnh quanh mỗi box cho phép thử DenseNet crop.
MAX_DISPLAY_BOXES = 60
assert 0 <= CROP_PAD_RATIO <= 1
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Lấy đúng FULL ZIP từ Google Drive

Đăng nhập tài khoản có quyền truy cập file. Ô này kiểm tra tên, thư mục, dung lượng và MD5 (nếu Drive cung cấp), rồi tải về bộ nhớ Colab. Lần chạy lại trong cùng runtime sẽ dùng bản đã xác thực.

In [2]:
# 2. Xác thực Drive và tải FULL ZIP (khoảng 779 MB)
def digest_file(path, algorithm="md5"):
    h = hashlib.new(algorithm)
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()

auth.authenticate_user()
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
drive = build("drive", "v3", credentials=creds, cache_discovery=False)
meta = drive.files().get(
    fileId=DRIVE_FILE_ID,
    fields="id,name,size,md5Checksum,mimeType,parents",
    supportsAllDrives=True,
).execute()
assert meta["name"] == ZIP_NAME, meta
assert int(meta["size"]) == EXPECTED_ZIP_BYTES, meta
assert DRIVE_FOLDER_ID in meta.get("parents", []), meta
assert meta["mimeType"] in ("application/zip", "application/x-zip-compressed"), meta
drive_md5 = meta.get("md5Checksum")

def valid_zip_cache():
    if not ZIP_PATH.is_file() or ZIP_PATH.stat().st_size != EXPECTED_ZIP_BYTES:
        return False
    return not drive_md5 or digest_file(ZIP_PATH) == drive_md5.lower()

if not valid_zip_cache():
    partial = ZIP_PATH.with_suffix(".zip.part")
    partial.unlink(missing_ok=True)
    request = drive.files().get_media(fileId=DRIVE_FILE_ID, supportsAllDrives=True)
    with partial.open("wb") as stream:
        downloader = MediaIoBaseDownload(stream, request, chunksize=8 << 20)
        done, last_decile = False, -1
        while not done:
            status, done = downloader.next_chunk(num_retries=3)
            if status is not None:
                decile = int(status.progress() * 10)
                if decile > last_decile:
                    print(f"Đã tải {status.progress():.0%}")
                    last_decile = decile
    assert partial.stat().st_size == EXPECTED_ZIP_BYTES, "ZIP tải về thiếu byte"
    if drive_md5:
        assert digest_file(partial) == drive_md5.lower(), "MD5 không khớp Drive"
    partial.replace(ZIP_PATH)

print("FULL ZIP:", ZIP_PATH)
print("Bytes:", ZIP_PATH.stat().st_size, "| SHA256:", digest_file(ZIP_PATH, "sha256"))


FULL ZIP: /content/exp034_demo/EXP-DENSENET121-YOLO-COMBINED-TRAINING-034_FULL.zip
Bytes: 779388067 | SHA256: 889017cc655a87eee2c64619ac6adeecf3336b795825da6b56f09a66a4cee7bb


## Giải nén bundle và nạp mô hình

Chỉ lấy `pipeline_bundle` từ FULL ZIP. Mã kiểm tra SHA256 của các file theo manifest đi kèm, rồi nạp 5 DenseNet121, detector YOLO và stacker F3.

In [3]:
# 3. Giải nén đúng bundle, chống đường dẫn ZIP không an toàn và kiểm tra SHA256
prefix = f"{EXPERIMENT_ID}/pipeline_bundle/"
with zipfile.ZipFile(ZIP_PATH) as archive:
    members = [info for info in archive.infolist()
               if info.filename.startswith(prefix) and not info.is_dir()]
    assert members, f"Không tìm thấy {prefix} trong ZIP"
    names = {info.filename[len(prefix):] for info in members}
    required = {
        "cashew_pipeline.py", "best.pt", "stacker.joblib",
        "frozen_config.json", "decision.json", "artifact_sha256.csv",
    } | {f"seed_{seed}/deployment_model.keras" for seed in (42, 123, 2026, 3407, 7777)}
    assert required <= names, f"Thiếu file bundle: {sorted(required - names)}"
    for info in members:
        rel = PurePosixPath(info.filename[len(prefix):])
        assert rel.parts and not rel.is_absolute() and ".." not in rel.parts, info.filename
        target = BUNDLE_DIR.joinpath(*rel.parts)
        assert target.resolve().is_relative_to(BUNDLE_DIR.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_name(target.name + ".part")
        with archive.open(info) as source, temporary.open("wb") as sink:
            shutil.copyfileobj(source, sink, length=8 << 20)
        temporary.replace(target)

with (BUNDLE_DIR / "artifact_sha256.csv").open(newline="", encoding="utf-8") as stream:
    manifest = list(csv.DictReader(stream))
assert manifest and {"path", "bytes", "sha256"} <= set(manifest[0])
verified = set()
self_rows = 0
for row in manifest:
    rel = PurePosixPath(row["path"])
    assert rel.parts and not rel.is_absolute() and ".." not in rel.parts
    item = BUNDLE_DIR.joinpath(*rel.parts)
    assert item.resolve().is_relative_to(BUNDLE_DIR.resolve())
    if rel.as_posix() == "artifact_sha256.csv":
        # The producer can rewrite this file after recording its old size/hash.
        # The ZIP member CRC was checked while extracting the complete stream.
        self_rows += 1
        continue
    assert row["path"] not in verified, f"Duplicate manifest row: {row['path']}"
    assert item.is_file() and item.stat().st_size == int(row["bytes"]), row["path"]
    assert digest_file(item, "sha256") == row["sha256"].lower(), row["path"]
    verified.add(row["path"])
assert self_rows <= 1, "Multiple self-referential manifest rows"
assert required - {"artifact_sha256.csv"} <= verified, "Required model missing from manifest"
print(f"Verified SHA256 for {len(verified)} bundle files.")
if self_rows:
    print("Skipped stale self-reference for artifact_sha256.csv.")
print("Manifest actual SHA256:", digest_file(BUNDLE_DIR / "artifact_sha256.csv", "sha256"))


Verified SHA256 for 10 bundle files.
Skipped stale self-reference for artifact_sha256.csv.
Manifest actual SHA256: c4b97d09b169ada9565fcfd7d793a2e6ba523e98aace00c6245f8311ac6c3eeb


In [4]:
# 4. Nạp DenseNet121, YOLO và F3 từ bundle
import joblib
from ultralytics import YOLO

module_path = BUNDLE_DIR / "cashew_pipeline.py"
spec = importlib.util.spec_from_file_location("exp034_cashew_pipeline", module_path)
pipeline_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pipeline_module
spec.loader.exec_module(pipeline_module)  # Đăng ký RandomHSVJitter cho Keras

config = json.loads((BUNDLE_DIR / "frozen_config.json").read_text(encoding="utf-8"))
decision = json.loads((BUNDLE_DIR / "decision.json").read_text(encoding="utf-8"))
assert decision["selected_default_rule"] == "F3", decision
classes = list(config["classes"])
detector_classes = list(config["detector_classes"])
seeds = list(config["seeds"])
assert len(seeds) == 5 and len(classes) == 5 and len(detector_classes) == 3
models = []
for seed in seeds:
    path = BUNDLE_DIR / f"seed_{seed}" / "deployment_model.keras"
    model = tf.keras.models.load_model(path, compile=False)
    assert model.input_shape[1:] == (256, 256, 3)
    assert model.output_shape[-1] == len(classes)
    models.append(model)
    print("Đã nạp DenseNet seed", seed)

detector = YOLO(str(BUNDLE_DIR / "best.pt"))
stackers = joblib.load(BUNDLE_DIR / "stacker.joblib")
assert "F3" in stackers
box_conf = float(config["box_conf"] if BOX_CONF_OVERRIDE is None else BOX_CONF_OVERRIDE)
assert 0 <= box_conf <= 1
print("Lớp DenseNet:", classes)
print("Lớp YOLO:", detector_classes)
print("Box hiển thị từ YOLO conf >=", box_conf, "| F3:", decision["selected_default_rule"])


Đã nạp DenseNet seed 42
Đã nạp DenseNet seed 123
Đã nạp DenseNet seed 2026
Đã nạp DenseNet seed 3407
Đã nạp DenseNet seed 7777
Lớp DenseNet: ['anthracnose', 'healthy', 'leaf_miner', 'red_rust', 'not_cashew_leaf']
Lớp YOLO: ['anthracnose', 'leaf_miner', 'red_rust']
Box hiển thị từ YOLO conf >= 0.25 | F3: F3


## Tải ảnh từ máy tính, không dùng upload widget

Trong Colab, mở biểu tượng **Files (thư mục)** ở thanh bên trái, bấm nút **Upload** ở đầu bảng Files hoặc kéo thả ảnh từ máy vào đó. Chờ ảnh xuất hiện ở `/content`, rồi chạy ô bên dưới. Bạn cũng có thể đưa ảnh vào `/content/exp034_demo/uploads`.

Ô này chỉ quét file JPG/PNG/WebP/BMP có sẵn trong hai thư mục; **không gọi `files.upload()`**. Nút Upload cũ đang hiện trong output của ô trước sẽ không dùng được sau khi đổi phiên trình duyệt. Sau khi tải thêm ảnh, chạy lại ô suy luận; nó sẽ tự quét ảnh mới.


In [ ]:
# 5. Doc anh da tai qua thanh Files cua Colab; khong dung upload widget.
ALLOWED_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def find_uploaded_images():
    candidates = {}
    for folder in (Path("/content"), UPLOAD_DIR):
        if not folder.is_dir():
            continue
        for source in sorted(folder.iterdir()):
            if not source.is_file() or source.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES:
                continue
            target = UPLOAD_DIR / source.name
            if source != target:
                shutil.copy2(source, target)
            try:
                with Image.open(target) as image:
                    image.verify()
            except Exception as exc:
                print("Bo qua file anh loi:", source.name, exc)
                continue
            candidates[target.name] = target
    return sorted(candidates.values(), key=lambda path: path.name.lower())

UPLOAD_PATHS = find_uploaded_images()
print("Anh san sang:", [path.name for path in UPLOAD_PATHS])
if not UPLOAD_PATHS:
    print("Chua co anh. Hay tai anh qua thanh Files vao /content, roi chay lai o nay.")


In [ ]:
# 6. Hàm suy luận: YOLO tìm box, DenseNet phân loại toàn ảnh và từng crop; F3 tính riêng
def iou_xyxy(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union else 0.0

def detect_boxes(rgb):
    # Ultralytics nhận numpy ảnh BGR; RGB của PIL được chuyển trước khi suy luận.
    bgr = rgb[:, :, ::-1].copy()
    result = detector.predict(source=bgr, conf=0.05, iou=0.7,
                              imgsz=640, rect=False, verbose=False)[0]
    boxes = []
    for cls_id, confidence, xyxy in zip(
        result.boxes.cls.cpu().numpy(),
        result.boxes.conf.cpu().numpy(),
        result.boxes.xyxy.cpu().numpy(),
    ):
        index = int(cls_id)
        assert 0 <= index < len(detector_classes), index
        boxes.append({"yolo_label": detector_classes[index],
                      "yolo_confidence": float(confidence),
                      "xyxy": [float(v) for v in xyxy]})
    return boxes

def select_display_boxes(raw_boxes):
    shown = []
    for box in sorted((b for b in raw_boxes if b["yolo_confidence"] >= box_conf),
                      key=lambda b: -b["yolo_confidence"]):
        if all(box["yolo_label"] != old["yolo_label"] or
               iou_xyxy(box["xyxy"], old["xyxy"]) <= 0.5 for old in shown):
            shown.append(box)
        if len(shown) >= MAX_DISPLAY_BOXES:
            break
    return shown

def padded_crop(rgb, xyxy):
    height, width = rgb.shape[:2]
    x1, y1, x2, y2 = xyxy
    pad_x = (x2 - x1) * CROP_PAD_RATIO
    pad_y = (y2 - y1) * CROP_PAD_RATIO
    left = max(0, int(np.floor(x1 - pad_x)))
    top = max(0, int(np.floor(y1 - pad_y)))
    right = min(width, int(np.ceil(x2 + pad_x)))
    bottom = min(height, int(np.ceil(y2 + pad_y)))
    if right <= left or bottom <= top:
        return None
    return rgb[top:bottom, left:right]

def dense_batch(images):
    # Model đã chứa Rescaling và Normalization; chỉ resize giống EXP-034.
    batch = np.stack([
        tf.image.resize(tf.convert_to_tensor(image, dtype=tf.float32),
                        (256, 256)).numpy()
        for image in images
    ])
    per_seed = np.stack([
        np.asarray(model.predict(batch, batch_size=8, verbose=0), dtype=float)
        for model in models
    ])
    mean_probs = per_seed.mean(axis=0)
    assert np.isfinite(mean_probs).all()
    return mean_probs

def fused_f3(whole_probs, raw_boxes, image_shape):
    height, width = image_shape[:2]
    evidence = {"max_conf": {}, "n_boxes": {}, "area_frac": {}}
    for cls in detector_classes:
        subset = [box for box in raw_boxes if box["yolo_label"] == cls]
        evidence["max_conf"][cls] = max(
            [box["yolo_confidence"] for box in subset], default=0.0)
        evidence["n_boxes"][cls] = len(subset)
        evidence["area_frac"][cls] = min(
            sum((b["xyxy"][2] - b["xyxy"][0]) *
                (b["xyxy"][3] - b["xyxy"][1]) for b in subset) / (width * height), 1.0)
    feature = np.array(
        list(whole_probs) +
        [evidence[kind][cls] for kind in ("max_conf", "n_boxes", "area_frac")
         for cls in detector_classes], dtype=float)
    stacker = stackers["F3"]
    fitted = stacker.predict_proba(feature[None, :])[0]
    probs = np.zeros(len(classes), dtype=float)
    probs[stacker[-1].classes_] = fitted
    temperature = float(config["temperature"]["F3"])
    probs = np.clip(probs, 1e-9, 1.0) ** (1.0 / temperature)
    probs /= probs.sum()
    best = int(np.argmax(probs))
    abstain = bool(probs[best] < config["abstain_thresholds"]["F3"])
    return {"label": classes[best], "confidence": float(probs[best]),
            "abstain": abstain, "probs": probs, "evidence": evidence}

def classify_image(path):
    with Image.open(path) as image:
        rgb = np.asarray(ImageOps.exif_transpose(image).convert("RGB"))
    raw_boxes = detect_boxes(rgb)
    shown = select_display_boxes(raw_boxes)
    valid = [(box, padded_crop(rgb, box["xyxy"])) for box in shown]
    valid = [(box, crop) for box, crop in valid if crop is not None]
    probabilities = dense_batch([rgb] + [crop for _, crop in valid])
    whole = probabilities[0]
    f3 = fused_f3(whole, raw_boxes, rgb.shape)
    boxes = []
    for (box, _), probs in zip(valid, probabilities[1:]):
        label_id = int(np.argmax(probs))
        boxes.append({**box, "densenet_crop_label": classes[label_id],
                      "densenet_crop_confidence": float(probs[label_id])})
    whole_id = int(np.argmax(whole))
    return {"rgb": rgb, "boxes": boxes, "raw_box_count": len(raw_boxes),
            "whole_dense_label": classes[whole_id],
            "whole_dense_confidence": float(whole[whole_id]),
            "whole_dense_probs": whole, "f3": f3}

def draw_result(rgb, boxes):
    image = Image.fromarray(rgb).copy()
    draw = ImageDraw.Draw(image)
    width, height = image.size
    font_size = max(16, min(width, height) // 48)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                                  font_size)
    except OSError:
        font = ImageFont.load_default()
    colors = {"anthracnose": "#e53935", "leaf_miner": "#1e88e5",
              "red_rust": "#f9a825"}
    for number, box in enumerate(boxes, 1):
        x1, y1, x2, y2 = [int(round(v)) for v in box["xyxy"]]
        x1, x2 = max(0, x1), min(width - 1, x2)
        y1, y2 = max(0, y1), min(height - 1, y2)
        color = colors.get(box["yolo_label"], "#ffffff")
        draw.rectangle((x1, y1, x2, y2), outline=color, width=max(3, font_size // 7))
        lines = [
            f"#{number} DenseNet crop: {box['densenet_crop_label']} "
            f"{box['densenet_crop_confidence']:.1%}",
            f"YOLO box: {box['yolo_label']} {box['yolo_confidence']:.1%}",
        ]
        line_height = font_size + 7
        text_width = max(draw.textbbox((0, 0), line, font=font)[2] for line in lines)
        label_y = max(0, y1 - 2 * line_height - 4)
        if label_y == 0 and y1 < 2 * line_height + 4:
            label_y = min(height - 2 * line_height - 4, y2 + 3)
        label_x = min(x1, max(0, width - text_width - 8))
        draw.rectangle((label_x, label_y, min(width - 1, label_x + text_width + 8),
                        min(height - 1, label_y + 2 * line_height + 2)),
                       fill="#111111")
        for index, line in enumerate(lines):
            draw.text((label_x + 4, label_y + index * line_height),
                      line, font=font, fill="#ffffff")
    return image


## Chạy trên ảnh đã tải

Mỗi box có hai dòng: **DenseNet crop** là nhãn/softmax phân loại crop, **YOLO box** là lớp/điểm định vị. Dòng tổng kết ghi riêng DenseNet toàn ảnh và F3 chính thức. Nếu YOLO không tìm thấy box trên ngưỡng Validation, ảnh vẫn được phân loại toàn ảnh và không vẽ box.

In [ ]:
# 7. Hiển thị ảnh, bảng điểm và lưu PNG/CSV
UPLOAD_PATHS = find_uploaded_images()  # Quét lại để thấy ảnh mới tải qua thanh Files.
if not UPLOAD_PATHS:
    raise RuntimeError("Chưa thấy ảnh trong /content hoặc /content/exp034_demo/uploads. Hãy tải qua thanh Files rồi chạy lại ô này.")
summary_rows = []
box_rows = []
for path in UPLOAD_PATHS:
    result = classify_image(path)
    f3 = result["f3"]
    print("\nẢnh:", path.name)
    print(f"DenseNet toàn ảnh (5 seed mean): {result['whole_dense_label']} "
          f"{result['whole_dense_confidence']:.1%} (softmax chưa hiệu chuẩn)")
    print(f"EXP-034 F3 chính thức: {f3['label']} {f3['confidence']:.1%}"
          + (" | CẦN CHUYÊN GIA KIỂM TRA (abstain)" if f3["abstain"] else ""))
    print(f"YOLO: {result['raw_box_count']} box thô; "
          f"{len(result['boxes'])} box hiển thị ở ngưỡng {box_conf:.2f}.")
    rendered = draw_result(result["rgb"], result["boxes"])
    display(rendered)
    output_png = OUTPUT_DIR / f"{path.stem}_annotated.png"
    rendered.save(output_png)
    summary_rows.append({
        "image": path.name,
        "densenet_whole_label": result["whole_dense_label"],
        "densenet_whole_softmax": result["whole_dense_confidence"],
        "f3_label": f3["label"], "f3_calibrated_confidence": f3["confidence"],
        "f3_abstain": f3["abstain"], "yolo_raw_boxes": result["raw_box_count"],
        "boxes_shown": len(result["boxes"]), "annotated_png": output_png.name,
    })
    for number, box in enumerate(result["boxes"], 1):
        box_rows.append({
            "image": path.name, "box_number": number,
            "x1": box["xyxy"][0], "y1": box["xyxy"][1],
            "x2": box["xyxy"][2], "y2": box["xyxy"][3],
            "yolo_box_label": box["yolo_label"],
            "yolo_box_confidence": box["yolo_confidence"],
            "densenet_crop_label": box["densenet_crop_label"],
            "densenet_crop_softmax": box["densenet_crop_confidence"],
        })

summary = pd.DataFrame(summary_rows)
details = pd.DataFrame(box_rows, columns=[
    "image", "box_number", "x1", "y1", "x2", "y2",
    "yolo_box_label", "yolo_box_confidence",
    "densenet_crop_label", "densenet_crop_softmax",
])
display(summary)
display(details)
summary.to_csv(OUTPUT_DIR / "image_summary.csv", index=False, encoding="utf-8-sig")
details.to_csv(OUTPUT_DIR / "box_details.csv", index=False, encoding="utf-8-sig")
print("Đã lưu kết quả tại:", OUTPUT_DIR)


In [ ]:
# 8. Tùy chọn: tải ảnh đã vẽ và hai bảng CSV về máy tính
archive_path = Path(shutil.make_archive(str(WORK_DIR / "exp034_uploaded_image_results"),
                                        "zip", root_dir=OUTPUT_DIR))
files.download(str(archive_path))
